In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("MyNotebook")
    .getOrCreate()
)

spark


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/11 15:53:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
taxi_trips = spark.read.format("delta").load("s3a://delta-data/silver/silver_taxi_trips")
taxi_zones = spark.read.format("delta").load("s3a://delta-data/silver/silver_taxi_zones")
air_quality = spark.read.format("delta").load("s3a://delta-data/silver/silver_air_quality")
weather = spark.read.format("delta").load("s3a://delta-data/silver/silver_weather")

26/09/11 15:53:45 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


taxi zone join

In [3]:
zones_drop = taxi_zones.withColumnsRenamed({
    "location_id": "dropoff_loc_id",
    "borough":"dropoff_borough",
    "zone": "dropoff_zone_name"  # adjust based on your actual column names
})  
zones_pick = taxi_zones.withColumnsRenamed({
    "location_id": "pickup_loc_id",
    "borough":"pickup_borough",
    "zone": "pickup_zone_name"
})


In [4]:
df = taxi_trips
df = df.join(zones_drop, df["dropoff_zone_id"] == zones_drop["dropoff_loc_id"], "left")
df = df.join(zones_pick, df["pickup_zone_id"] == zones_pick["pickup_loc_id"], "left")

In [5]:
df = df.drop("dropoff_zone_id", "pickup_zone_id","pickup_loc_id","dropoff_loc_id")


weather

In [6]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col


In [7]:
weather_pu = weather.select(
    col("temp").alias("temp_pu"),
    col("rhum").alias("rhum_pu"),
    col("prcp").alias("prcp_pu"),
    col("snwd").alias("snwd_pu"),
    col("wdir").alias("wdir_pu"),
    col("wspd").alias("wspd_pu"),
    col("wpgt").alias("wpgt_pu"),
    col("pres").alias("pres_pu"),
    col("cldc").alias("cldc_pu"),
    col("observation_datetime").alias("observation_datetime_pu"),
    col("coco").alias("coco_pu")
)


weather_do = weather.select(
    col("temp").alias("temp_do"),
    col("rhum").alias("rhum_do"),
    col("prcp").alias("prcp_do"),
    col("snwd").alias("snwd_do"),
    col("wdir").alias("wdir_do"),
    col("wspd").alias("wspd_do"),
    col("wpgt").alias("wpgt_do"),
    col("pres").alias("pres_do"),
    col("cldc").alias("cldc_do"),
    col("observation_datetime").alias("observation_datetime_do"),
    col("coco").alias("coco_do")
)
weather_pu.show()
weather_do.show()

26/09/11 15:53:50 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-------+-------+-------+-------+-------+-------+-------+-------+-----------------------+-------+
|temp_pu|rhum_pu|prcp_pu|snwd_pu|wdir_pu|wspd_pu|wpgt_pu|pres_pu|cldc_pu|observation_datetime_pu|coco_pu|
+-------+-------+-------+-------+-------+-------+-------+-------+-------+-----------------------+-------+
|    6.1|     47|    0.0|   NULL|    280|    9.4|   NULL| 1015.8|      8|    2024-01-01 00:00:00|      3|
|    6.1|     45|    0.0|   NULL|    260|   13.0|   NULL| 1015.8|      7|    2024-01-01 01:00:00|      3|
|    6.1|     47|    0.0|   NULL|    250|   13.0|   NULL| 1015.7|      8|    2024-01-01 02:00:00|      3|
|    6.1|     51|    0.0|   NULL|    260|   13.0|   NULL| 1015.7|      8|    2024-01-01 03:00:00|      4|
|    6.1|     51|    0.0|   NULL|    250|   13.0|   NULL| 1015.7|      8|    2024-01-01 04:00:00|      4|
|    6.7|     49|    0.0|   NULL|    240|   14.8|   NULL| 1015.8|      8|    2024-01-01 05:00:00|      4|
|    6.1|     53|    0.0|   NULL|    230|   11

In [8]:
df = df.withColumn(
    "do_datetime_hour",
    F.date_trunc("hour", F.col("dropoff_datetime"))
)
df = df.withColumn(
    "pu_datetime_hour",
    F.date_trunc("hour", F.col("pickup_datetime"))
)

In [9]:
df = df.join(
    weather_do
    ,df["do_datetime_hour"] == weather_do["observation_datetime_do"]
    ,"left"
)

df = df.join(
    weather_pu
    ,df["pu_datetime_hour"] == weather_pu["observation_datetime_pu"]
    ,"left"
)


In [10]:
df = df.drop("observation_datetime_pu", "observation_datetime_do")

air quality data

In [11]:
df_aq= air_quality.filter(air_quality.state_name == "New York")
df_aq = df_aq.drop("date_local","time_local","time_gmt","date_gmt","site_num","state_name","Latitude","Longitude","parameter_code","state_code","county_code","POC","Datum","county_name","method_code","date_of_last_change","observation_id")


In [12]:
df_aq = df_aq.groupBy("observation_datetime").agg(
    F.avg("sample_measurement").alias("average_value"),
    F.avg("MDL").alias("MDL"),
    F.concat_ws(", ", F.collect_list("Qualifier")).alias("Qualifiers"),
    F.max("parameter_name").alias("parameter_name"), # it is constant between the values
    F.max("units_of_measure").alias("units_of_measure"), # it is constant between the values
    F.max("Uncertainty").alias("Uncertainty"), # it is constant between the values
    F.max("method_name").alias("method_name"), # it is constant between the values
    F.max("method_type").alias("method_type") # it is constant between the values
)


In [13]:
aq_pu = df_aq.select(
    col("average_value").alias("average_value_pu"),
    col("MDL").alias("MDL_pu"),
    col("observation_datetime").alias("observation_datetime_pu"),
    col("Qualifiers").alias("Qualifiers_pu"),
    col("parameter_name").alias("parameter_name_pu"),
    col("units_of_measure").alias("units_of_measure_pu"),
    col("Uncertainty").alias("Uncertainty_pu"),
    col("method_name").alias("method_name_pu"),
    col("method_type").alias("method_type_pu")
)
aq_do = df_aq.select(
    col("average_value").alias("average_value_do"),
    col("MDL").alias("MDL_do"),
    col("observation_datetime").alias("observation_datetime_do"),
    col("Qualifiers").alias("Qualifiers_do"),
    col("parameter_name").alias("parameter_name_do"),
    col("units_of_measure").alias("units_of_measure_do"),
    col("Uncertainty").alias("Uncertainty_do"),
    col("method_name").alias("method_name_do"),
    col("method_type").alias("method_type_do")
)

In [14]:
df = df.join(
    aq_pu
    ,df["pu_datetime_hour"] == aq_pu["observation_datetime_pu"]
    ,"left"
)

df = df.join(
    aq_do
    ,df["do_datetime_hour"] == aq_do["observation_datetime_do"]
    ,"left"
)

In [16]:
df.show()


+---------+-------------------+-------------------+---------------+-------------+------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+---------------+--------------------+------------+--------------+--------------------+------------+-------------------+-------------------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-----------------+------+-----------------------+-------------+--------------------+--------------------+--------------+--------------------+--------------+-----------------+------+-----------------------+-------------+--------------------+--------------------+--------------+--------------------+--------------+
|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|trip_distance|rate_code_id|store_and_fwd_flag|payment_type|fare_a

26/09/11 16:10:33 ERROR StandaloneSchedulerBackend: Application has been killed. Reason: Master removed our application: KILLED
26/09/11 16:10:33 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exiting due to error from cluster scheduler: Master removed our application: KILLED
	at org.apache.spark.errors.SparkCoreErrors$.clusterSchedulerError(SparkCoreErrors.scala:295)
	at org.apache.spark.scheduler.TaskSchedulerImpl.error(TaskSchedulerImpl.scala:955)
	at org.apache.spark.scheduler.cluster.StandaloneSchedulerBackend.dead(StandaloneSchedulerBackend.scala:166)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint.markDead(StandaloneAppClient.scala:273)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint$$anonfun$receive$1.applyOrElse(StandaloneAppClient.scala:174)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:116)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:216)
	at org.apache.spark.rpc.netty.Inbox.proce